# Chat with PDF - Agent Framework Workflow version

This notebook replicates the promptflow "Chat with PDF" DAG flow using **Microsoft Agent Framework's `WorkflowBuilder`**.
Each promptflow node is mapped to an `Executor`, and `WorkflowBuilder.add_edge()` enforces the same fixed execution order:

```
download → build_index → rewrite_question → find_context → qna
```

## 0. Install dependencies

In [5]:
%pip install -r requirements.txt

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached faiss_cpu-1.13.2-cp311-cp311-win_amd64.whl.metadata (7.6 kB)
  Using cached openai-2.31.0-py3-none-any.whl.metadata (31 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached tiktoken-0.12.0-cp311-cp311-win_amd64.whl.metadata (6.9 kB)
  Using cached azure_ai_projects-2.0.1-py3-none-any.whl.metadata (89 kB)
  Using cached azure_ai_agents-1.2.0b5-py3-none-any.whl.metadata (74 kB)
  Using cached azure_ai_inference-1.0.0b9-py3-none-any.whl.metadata (34 kB)
  Using cached azure_identity-1.25.3-py3-none-any.whl.metadata (91 kB)
  Using cached aiohttp-3.13.5-cp311-cp311-win_amd64.whl.metadata (8.4 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenl

## 1. Configure environment

Copy `.env.template` to `.env` and fill in your Azure OpenAI (or OpenAI) credentials.
The agent reads configuration from environment variables instead of promptflow connections.

In [13]:
import os, sys
sys.path.insert(0, os.path.abspath("."))

from dotenv import load_dotenv
load_dotenv(override=False)

# Verify key env vars are set
for var in ["OPENAI_API_KEY", "OPENAI_API_BASE", "CHAT_MODEL_DEPLOYMENT_NAME", "EMBEDDING_MODEL_DEPLOYMENT_NAME"]:
    val = os.environ.get(var)
    print(f"{var}: {'SET' if val and not val.startswith('<') else 'NOT SET - update .env'}")

OPENAI_API_KEY: NOT SET - update .env
OPENAI_API_BASE: SET
CHAT_MODEL_DEPLOYMENT_NAME: SET
EMBEDDING_MODEL_DEPLOYMENT_NAME: SET


## 2. Test the workflow agent

Run a single question through the fixed workflow pipeline.
The `WorkflowAgent` executes: `download → build_index → rewrite_question → find_context → qna` in order.

In [14]:
from chat_with_pdf_agent import chat_with_pdf

output = await chat_with_pdf(
    question="what is BERT?",
    pdf_url="https://arxiv.org/pdf/1810.04805.pdf",
    chat_history=[],
)
print(output["answer"])

Pdf already exists in c:\Users\lusu\code\promptflow\examples\flows\chat\chat-with-pdf-agent\chat_with_pdf\.pdfs\https___arxiv.org_pdf_1810.04805.pdf.pdf
Chunk size: 1024, chunk overlap: 64
Index path: C:\Users\lusu\code\promptflow\examples\flows\chat\chat-with-pdf-agent\chat_with_pdf\.index\.pdfs\https___arxiv.org_pdf_1810.04805.pdf.pdf.index_1024_64
Index already exists, bypassing index creation
Func execution failed. Retrying in 5 seconds: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}
Func execution failed. Retrying in 5 seconds: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}
Fu

Exception: Func execution failed after 5 retries: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

## 3. Run batch over a data file

Equivalent to `pf.run(flow=flow_path, data=data_path, ...)` — processes each line in the JSONL file.

In [11]:
import json
from chat_with_pdf_agent import chat_with_pdf

data_path = "./data/bert-paper-qna-3-line.jsonl"

config_2k_context = {
    "EMBEDDING_MODEL_DEPLOYMENT_NAME": os.environ.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-ada-002"),
    "CHAT_MODEL_DEPLOYMENT_NAME": os.environ.get("CHAT_MODEL_DEPLOYMENT_NAME", "gpt-4"),
    "PROMPT_TOKEN_LIMIT": 2000,
    "MAX_COMPLETION_TOKENS": 256,
    "VERBOSE": True,
    "CHUNK_SIZE": 1024,
    "CHUNK_OVERLAP": 64,
}

results_2k = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        result = await chat_with_pdf(
            question=row["question"],
            pdf_url=row["pdf_url"],
            chat_history=row.get("chat_history", []),
            config=config_2k_context,
        )
        results_2k.append({"question": row["question"], **result})
        print(f"Q: {row['question']}")
        print(f"A: {result['answer'][:200]}...\n")

print(f"Processed {len(results_2k)} rows")

Pdf already exists in c:\Users\lusu\code\promptflow\examples\flows\chat\chat-with-pdf-agent\chat_with_pdf\.pdfs\https___arxiv.org_pdf_1810.04805.pdf.pdf
Chunk size: 1024, chunk overlap: 64
Index path: C:\Users\lusu\code\promptflow\examples\flows\chat\chat-with-pdf-agent\chat_with_pdf\.index\.pdfs\https___arxiv.org_pdf_1810.04805.pdf.pdf.index_1024_64
Index already exists, bypassing index creation
Func execution failed. Retrying in 5 seconds: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}
Func execution failed. Retrying in 5 seconds: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}
Fu

Exception: Func execution failed after 5 retries: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

## 4. View results

Equivalent to `pf.get_details()` — display the batch results.

In [ ]:
for i, r in enumerate(results_2k):
    print(f"--- Row {i+1} ---")
    print(f"Question: {r['question']}")
    print(f"Answer: {r['answer']}")
    print(f"Context snippets: {len(r['context'])}")
    print()

## 5. Try a different configuration (3k context) and compare

Equivalent to the promptflow experimentation step — run with a different PROMPT_TOKEN_LIMIT.

In [ ]:
config_3k_context = {
    "EMBEDDING_MODEL_DEPLOYMENT_NAME": os.environ.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-ada-002"),
    "CHAT_MODEL_DEPLOYMENT_NAME": os.environ.get("CHAT_MODEL_DEPLOYMENT_NAME", "gpt-4"),
    "PROMPT_TOKEN_LIMIT": 3000,
    "MAX_COMPLETION_TOKENS": 256,
    "VERBOSE": True,
    "CHUNK_SIZE": 1024,
    "CHUNK_OVERLAP": 64,
}

results_3k = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        result = await chat_with_pdf(
            question=row["question"],
            pdf_url=row["pdf_url"],
            chat_history=row.get("chat_history", []),
            config=config_3k_context,
        )
        results_3k.append({"question": row["question"], **result})
        print(f"Q: {row['question']}")
        print(f"A: {result['answer'][:200]}...\n")

print(f"Processed {len(results_3k)} rows")

## 6. Compare results side by side

In [ ]:
for i in range(len(results_2k)):
    print(f"=== Question {i+1}: {results_2k[i]['question']} ===")
    print(f"\n[2k context] {results_2k[i]['answer'][:300]}")
    print(f"\n[3k context] {results_3k[i]['answer'][:300]}")
    print(f"\n[2k context snippets: {len(results_2k[i]['context'])}] vs [3k context snippets: {len(results_3k[i]['context'])}]")
    print("\n" + "="*80 + "\n")